# Create primary-key registry
Defines which tables to validate and the columns that form each table's primary key.

In [0]:
%sql
-- Keep configuration tables separate from the test data in dev
create schema if not exists validation_sweep.metadata;

In [0]:
%sql
-- Create the primary-key registry
create or replace table validation_sweep.metadata.pk_registry (
  catalog_name  string        not null,
  schema_name   string        not null,
  table_name    string        not null,
  key_columns   array<string> not null,
  is_active     boolean       not null,
  updated_at    timestamp     not null
);

-- Register development tables and their primary-key columns
insert into validation_sweep.metadata.pk_registry values
  ('validation_sweep', 'dev', 'customer', array('c_custkey'),                true, current_timestamp()),
  ('validation_sweep', 'dev', 'partsupp', array('ps_partkey', 'ps_suppkey'), true, current_timestamp());

In [0]:
# Verify that registered key columns exist in their development tables; report all errors before failing
registry = spark.table("validation_sweep.metadata.pk_registry").collect()
errors = []

for r in registry:
    full_name = f"{r.catalog_name}.{r.schema_name}.{r.table_name}"
    columns = spark.table(full_name).columns
    missing = [c for c in r.key_columns if c not in columns]

    if missing:
        errors.append(f"{full_name}: missing key columns {missing}")
        print(f"{full_name:35} {r.key_columns}  FAIL")
    else:
        print(f"{full_name:35} {r.key_columns}  OK")

if errors:
    raise ValueError("\n".join(errors))